In [1]:
suppressPackageStartupMessages({
  library(lme4)
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(emmeans)
  library(car)
  library(performance)
  library(DHARMa)
  library(AICcmodavg)
})

Warning message:
“package ‘lme4’ was built under R version 4.6.1”


Warning message:
“package ‘emmeans’ was built under R version 4.6.1”


Warning message:
“package ‘performance’ was built under R version 4.6.1”


# Soil Sediment Geochemistry - GLMM Analysis Pipeline
### Notebook 02 - Gamma GLMM Modelling (log link)

**Primary model per nutrient** (per analysis plan - treatment, sampling period and treatment x period are assigned fixed factors):

```
response ~ treatment * period + (1 | plot)        family = Gamma(link = "log")
```

- `plot` = random intercept (same 9 plots resampled each period).
- Primary model = **full model including the interaction**, regardless of the AIC comparison.
- Produces per nutrient: Wald Type III chi-squared tests, fixed-effect CIs, ICC + Nakagawa R-squared,
  `emmeans` response-scale means and contrasts, DHARMa diagnostics, and an informational model-comparison table.
- Treatment labels kept as recorded per sheet (`C`/`I` in Carbon; `c`/`i` in Nitrogen and Phosphorus);
  the control-type label is always the reference level.


In [2]:
soil <- readRDS("soil_tidied.rds")
nut_order <- c("Carbon", "Nitrogen", "Phosphorus")
nut_lab   <- c(Carbon = "SOC", Nitrogen = "Total N", Phosphorus = "Total P")
trt_cmp   <- c(Carbon = "C/I", Nitrogen = "c/i", Phosphorus = "c/i")
cat("Data loaded:", nrow(soil), "rows. Nutrients:", paste(nut_order, collapse = ", "), "\n")

Data loaded: 144 rows. Nutrients: Carbon, Nitrogen, Phosphorus 


## 1 - Fit the full Gamma GLMMs


In [3]:
fit_capture <- function(formula, df, bobyqa = FALSE) {
  warns <- character(0)
  ctrl  <- if (bobyqa) glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 100000)) else glmerControl()
  m <- withCallingHandlers(
    suppressMessages(
      glmer(formula, data = df, family = Gamma(link = "log"), control = ctrl)
    ),
    warning = function(w) {
      warns <<- c(warns, conditionMessage(w))
      invokeRestart("muffleWarning")
    }
  )
  attr(m, "fit_warnings") <- warns
  m
}

fit_one <- function(df) {
  f_full <- response ~ treatment * period + (1 | plot)
  m  <- fit_capture(f_full, df)
  if (length(attr(m, "fit_warnings")) > 0) {
    m2 <- fit_capture(f_full, df, bobyqa = TRUE)
    if (length(attr(m2, "fit_warnings")) == 0) {
      attr(m2, "fit_note") <- "refit with bobyqa"
      m <- m2
    }
  }
  m
}

fits <- setNames(lapply(nut_order, function(nm) fit_one(soil %>% filter(nutrient == nm))), nut_order)

fit_status <- data.frame(
  nutrient    = nut_order,
  class       = sapply(fits, function(m) class(m)[1]),
  n_warnings  = sapply(fits, function(m) length(attr(m, "fit_warnings"))),
  note        = sapply(fits, function(m) if (!is.null(attr(m, "fit_note"))) attr(m, "fit_note") else ""),
  warning_txt = sapply(fits, function(m) paste(gsub("\n", " ", attr(m, "fit_warnings")), collapse = " | "))
)
print(fit_status)

             nutrient    class n_warnings note warning_txt
Carbon         Carbon glmerMod          0                 
Nitrogen     Nitrogen glmerMod          0                 
Phosphorus Phosphorus glmerMod          0                 


## 2 - Fixed effects (log scale; exp() column interpretable for main-effect rows)


In [4]:
fx_tab <- bind_rows(lapply(nut_order, function(nm) {
  m   <- fits[[nm]]
  est <- fixef(m)
  se  <- sqrt(diag(vcov(m)))
  z   <- est / se
  data.frame(
    nutrient     = nm,
    term         = names(est),
    estimate_log = as.numeric(est),
    ci_low       = as.numeric(est) - 1.96 * as.numeric(se),
    ci_high      = as.numeric(est) + 1.96 * as.numeric(se),
    z            = as.numeric(z),
    p_value      = 2 * pnorm(-abs(as.numeric(z)))
  )
})) %>%
  mutate(exp_estimate = exp(estimate_log),
         across(c(estimate_log, ci_low, ci_high, z, p_value, exp_estimate), ~ signif(.x, 3)))
print(as.data.frame(fx_tab))

cat("\nMultiplicative interpretation: exp(estimate) is the multiplicative change on the response for that term,\nholding the other term at its reference level. Interaction rows are log-scale effect modifications.\n")

     nutrient                    term estimate_log  ci_low ci_high       z
1      Carbon             (Intercept)      1.32000  1.0800   1.560 10.8000
2      Carbon              treatmentI      0.00930 -0.1940   0.213  0.0896
3      Carbon            periodWeek_3     -0.06880 -0.2720   0.134 -0.6630
4      Carbon            periodWeek_6     -0.40500 -0.6090  -0.201 -3.8900
5      Carbon            periodWeek_9     -0.08230 -0.2860   0.121 -0.7930
6      Carbon treatmentI:periodWeek_3      0.11100 -0.1760   0.399  0.7600
7      Carbon treatmentI:periodWeek_6      0.19200 -0.0966   0.480  1.3000
8      Carbon treatmentI:periodWeek_9      0.02080 -0.2670   0.308  0.1420
9    Nitrogen             (Intercept)     -1.50000 -1.8200  -1.180 -9.2100
10   Nitrogen              treatmenti      0.20400 -0.1090   0.516  1.2800
11   Nitrogen            periodWeek_9     -0.43100 -0.7450  -0.117 -2.6900
12   Nitrogen treatmenti:periodWeek_9      0.14600 -0.3010   0.592  0.6400
13 Phosphorus            


Multiplicative interpretation: exp(estimate) is the multiplicative change on the response for that term,
holding the other term at its reference level. Interaction rows are log-scale effect modifications.


## 3 - Variance structure: random-effect SD, residual sigma (approx. CV), ICC, Nakagawa R-squared


In [5]:
vc_tab <- bind_rows(lapply(nut_order, function(nm) {
  m <- fits[[nm]]
  v <- as.data.frame(VarCorr(m))
  data.frame(
    nutrient      = nm,
    grp           = v$grp,
    name          = v$var1,
    sd            = round(v$sdcor, 4),
    residual_sd   = round(sigma(m), 4),
    icc_linkscale = round(v$sdcor[1]^2 / (v$sdcor[1]^2 + sigma(m)^2), 3)
  )
}))
print(as.data.frame(vc_tab))

r2_tab <- bind_rows(lapply(nut_order, function(nm) {
  m <- fits[[nm]]
  res <- tryCatch({
    rr  <- performance::r2_nakagawa(m, ci = FALSE, by_group = FALSE)
    nms <- names(rr)
    cnd <- nms[grepl("cond", nms, ignore.case = TRUE)][1]
    mrg <- nms[grepl("marg", nms, ignore.case = TRUE)][1]
    data.frame(nutrient = nm,
               R2_marginal    = if (!is.na(mrg)) as.numeric(rr[[mrg]][1]) else NA_real_,
               R2_conditional = if (!is.na(cnd)) as.numeric(rr[[cnd]][1]) else NA_real_)
  }, error = function(e) data.frame(nutrient = nm, R2_marginal = NA_real_, R2_conditional = NA_real_))
  res
})) %>%
  mutate(across(c(R2_marginal, R2_conditional), ~ round(.x, 3)))
print(as.data.frame(r2_tab))
cat("\nICC and Nakagawa R-squared are link-scale (log) summaries. sigma(m) is the Gamma residual scale parameter\n(approx. the coefficient of variation when variance ~ mean^2).\n")

    nutrient  grp        name     sd residual_sd icc_linkscale
1     Carbon plot (Intercept) 0.6314      0.2362         0.877
2   Nitrogen plot (Intercept) 0.6120      0.3570         0.746
3 Phosphorus plot (Intercept) 0.4275      0.1977         0.824


    nutrient R2_marginal R2_conditional
1     Carbon          NA             NA
2   Nitrogen          NA             NA
3 Phosphorus          NA             NA



ICC and Nakagawa R-squared are link-scale (log) summaries. sigma(m) is the Gamma residual scale parameter
(approx. the coefficient of variation when variance ~ mean^2).


## 4 - Wald Type III chi-squared tests (primary significance table)


In [6]:
anova_tab <- bind_rows(lapply(nut_order, function(nm) {
  a <- car::Anova(fits[[nm]], type = "III")
  data.frame(nutrient = nm,
             term     = rownames(a),
             chisq    = as.numeric(a$Chisq),
             df       = as.numeric(a$Df),
             p_value  = as.numeric(a[["Pr(>Chisq)"]]))
}))
print(as.data.frame(anova_tab %>% mutate(across(c(chisq, p_value), ~ signif(.x, 3)))))

cat("\nVerdict (alpha = 0.05):\n")
for (nm in nut_order) {
  a   <- anova_tab %>% filter(nutrient == nm, term != "(Intercept)")
  sig <- a$term[a$p_value < 0.05]
  ip  <- a$p_value[a$term == "treatment:period"]
  cat(sprintf("  %-10s significant fixed terms: %s | interaction p = %s -> %s\n",
              nut_lab[[nm]],
              if (length(sig)) paste(sig, collapse = ", ") else "none",
              ifelse(is.na(ip), "NA", sprintf("%.4g", ip)),
              ifelse(is.na(ip), "not estimable",
                     ifelse(ip < 0.05, "INTERACTION retained", "no detectable interaction"))))
}

     nutrient             term    chisq df  p_value
1      Carbon      (Intercept) 1.17e+02  1 3.05e-27
2      Carbon        treatment 8.03e-03  1 9.29e-01
3      Carbon           period 1.80e+01  3 4.33e-04
4      Carbon treatment:period 2.16e+00  3 5.40e-01
5    Nitrogen      (Intercept) 8.47e+01  1 3.41e-20
6    Nitrogen        treatment 1.63e+00  1 2.01e-01
7    Nitrogen           period 7.26e+00  1 7.07e-03
8    Nitrogen treatment:period 4.10e-01  1 5.22e-01
9  Phosphorus      (Intercept) 2.22e+03  1 0.00e+00
10 Phosphorus        treatment 1.46e+00  1 2.27e-01
11 Phosphorus           period 7.00e-01  1 4.03e-01
12 Phosphorus treatment:period 2.78e-03  1 9.58e-01



Verdict (alpha = 0.05):


  SOC        significant fixed terms: period | interaction p = 0.5399 -> no detectable interaction
  Total N    significant fixed terms: period | interaction p = 0.5221 -> no detectable interaction
  Total P    significant fixed terms: none | interaction p = 0.958 -> no detectable interaction


## 5 - Model comparison (informational)

The full model (`treatment * period`) remains the primary model per the analysis plan.
AIC/AICc/BIC against the additive model and the intercept-only null model are reported for context only.


In [7]:
variants <- setNames(lapply(nut_order, function(nm) {
  d <- soil %>% filter(nutrient == nm)
  list(full     = fits[[nm]],
       additive = fit_capture(response ~ treatment + period + (1 | plot), d),
       null     = fit_capture(response ~ 1 + (1 | plot), d))
}), nut_order)

cmp_tab <- bind_rows(lapply(nut_order, function(nm) {
  mv <- variants[[nm]]
  ml <- c(full     = "treatment * period (full)",
          additive = "treatment + period (additive)",
          null     = "intercept only (null)")
  data.frame(
    nutrient = nm,
    model    = unname(ml),
    k        = sapply(mv, function(m) attr(logLik(m), "df")),
    logLik   = sapply(mv, function(m) as.numeric(logLik(m))),
    AIC      = sapply(mv, AIC),
    AICc     = sapply(mv, function(m) AICcmodavg::AICc(m)),
    BIC      = sapply(mv, BIC)
  )
})) %>%
  group_by(nutrient) %>%
  mutate(dAIC_vs_full = round(AIC - AIC[1], 2)) %>%
  ungroup() %>%
  mutate(across(c(logLik, AIC, AICc, BIC), ~ round(.x, 2)))
print(as.data.frame(cmp_tab))

    nutrient                         model  k  logLik    AIC   AICc    BIC
1     Carbon     treatment * period (full) 10  -88.54 197.08 200.69 219.85
2     Carbon treatment + period (additive)  7  -89.60 193.21 194.96 209.14
3     Carbon         intercept only (null)  3  -99.66 205.31 205.66 212.14
4   Nitrogen     treatment * period (full)  6   40.11 -68.23 -65.33 -58.73
5   Nitrogen treatment + period (additive)  5   39.91 -69.82 -67.82 -61.90
6   Nitrogen         intercept only (null)  3   33.75 -61.51 -60.76 -56.76
7 Phosphorus     treatment * period (full)  6 -128.56 269.12 272.02 278.62
8 Phosphorus treatment + period (additive)  5 -128.56 267.12 269.12 275.04
9 Phosphorus         intercept only (null)  3 -130.56 267.12 267.87 271.87
  dAIC_vs_full
1         0.00
2        -3.87
3         8.23
4         0.00
5        -1.59
6         6.72
7         0.00
8        -2.00
9        -2.00


## 6 - Estimated marginal means (response scale) and contrasts

EMMs are back-transformed to the response scale (`type = "response"`), i.e. geometric-mean-type summaries
under the log link. Contrasts are impact-type vs control-type ratios within each period
(`I/C` in Carbon; `i/c` in Nitrogen and Phosphorus), holm-adjusted across the pairwise set.


In [8]:
emm_list <- lapply(fits, function(m) emmeans(m, ~ treatment | period, type = "response"))

emm_tab <- bind_rows(lapply(nut_order, function(nm) {
  d <- as.data.frame(summary(emm_list[[nm]], level = 0.95, infer = c(TRUE, TRUE)))
  names(d)[names(d) %in% c("response", "emmean", "rate")] <- "mean"
  names(d)[names(d) == "asymp.LCL"] <- "lower.CL"
  names(d)[names(d) == "asymp.UCL"] <- "upper.CL"
  d$nutrient <- nm
  d
})) %>%
  select(any_of(c("nutrient", "period", "treatment", "mean", "SE", "lower.CL", "upper.CL"))) %>%
  mutate(across(any_of(c("mean", "SE", "lower.CL", "upper.CL")), ~ signif(.x, 3)))
print(as.data.frame(emm_tab))

     nutrient period treatment   mean     SE lower.CL upper.CL
1      Carbon Week_0         C  3.740 0.4570    2.950    4.760
2      Carbon Week_0         I  3.780 0.4610    2.970    4.800
3      Carbon Week_3         C  3.490 0.4270    2.750    4.440
4      Carbon Week_3         I  3.940 0.4820    3.100    5.010
5      Carbon Week_6         C  2.500 0.3060    1.960    3.170
6      Carbon Week_6         I  3.050 0.3740    2.400    3.880
7      Carbon Week_9         C  3.450 0.4210    2.710    4.380
8      Carbon Week_9         I  3.550 0.4340    2.800    4.510
9    Nitrogen Week_0         c  0.224 0.0364    0.163    0.308
10   Nitrogen Week_0         i  0.274 0.0445    0.200    0.377
11   Nitrogen Week_9         c  0.145 0.0239    0.105    0.201
12   Nitrogen Week_9         i  0.206 0.0337    0.150    0.284
13 Phosphorus Week_0         c 41.100 3.2500   35.300   48.000
14 Phosphorus Week_0         i 36.800 2.9100   31.500   43.000
15 Phosphorus Week_9         c 44.400 3.5300   38.000  

In [9]:
pair_tab <- bind_rows(lapply(nut_order, function(nm) {
  s <- as.data.frame(summary(pairs(emm_list[[nm]]), infer = c(TRUE, TRUE), type = "response", adjust = "holm"))
  s$nutrient <- nm
  s
}))
estcol <- intersect(names(pair_tab), c("ratio", "effect", "odds.ratio"))[1]
if (is.na(estcol)) estcol <- names(pair_tab)[sapply(pair_tab, is.numeric)][1]
names(pair_tab)[names(pair_tab) == estcol] <- "ratio"
names(pair_tab)[names(pair_tab) %in% c("asymp.LCL", "lower.CL", "lower")] <- "LCL"
names(pair_tab)[names(pair_tab) %in% c("asymp.UCL", "upper.CL", "upper")] <- "UCL"
pair_tab <- pair_tab %>%
  select(any_of(c("nutrient", "contrast", "period", "ratio", "SE", "LCL", "UCL", "p.value"))) %>%
  mutate(across(c(ratio, SE, LCL, UCL, p.value), ~ signif(.x, 3)))
print(as.data.frame(pair_tab))

cat("\nTreatment-ratio verdict (alpha = 0.05):\n")
for (nm in nut_order) {
  t   <- pair_tab %>% filter(nutrient == nm)
  sig <- t %>% filter(p.value < 0.05)
  cat(sprintf("  %-10s (%s): %d comparisons, %d significant.\n",
              nut_lab[[nm]], trt_cmp[[nm]], nrow(t), nrow(sig)))
  for (i in seq_len(nrow(sig))) {
    cat(sprintf("      %s: ratio = %.3f [%.3f, %.3f], adj p = %.4g\n",
                sig$period[i], sig$ratio[i], sig$LCL[i], sig$UCL[i], sig$p.value[i]))
  }
}

    nutrient contrast period ratio     SE   LCL   UCL p.value
1     Carbon    C / I Week_0 0.991 0.1030 0.808 1.210  0.9290
2     Carbon    C / I Week_3 0.886 0.0919 0.723 1.090  0.2440
3     Carbon    C / I Week_6 0.818 0.0855 0.666 1.000  0.0544
4     Carbon    C / I Week_9 0.970 0.1010 0.792 1.190  0.7720
5   Nitrogen    c / i Week_0 0.816 0.1300 0.597 1.110  0.2010
6   Nitrogen    c / i Week_9 0.705 0.1160 0.511 0.973  0.0337
7 Phosphorus    c / i Week_0 1.120 0.1030 0.933 1.340  0.2270
8 Phosphorus    c / i Week_9 1.110 0.1020 0.927 1.330  0.2580



Treatment-ratio verdict (alpha = 0.05):


  SOC        (C/I): 4 comparisons, 0 significant.
  Total N    (c/i): 2 comparisons, 1 significant.
      Week_9: ratio = 0.705 [0.511, 0.973], adj p = 0.0337
  Total P    (c/i): 2 comparisons, 0 significant.


## 7 - DHARMa diagnostics (simulated scaled residuals)

`simulateResiduals` refits the model to generate the residual distribution expected under the fitted model;
KS non-uniformity, dispersion and outlier tests are reported. Visualised in Notebook 03.


In [10]:
sims <- lapply(fits, function(m) suppressMessages(simulateResiduals(m, n = 1000, seed = 2026)))

dh_tab <- bind_rows(lapply(nut_order, function(nm) {
  tt <- testResiduals(sims[[nm]], plot = FALSE)
  data.frame(
    nutrient     = nm,
    KS_p         = as.numeric(tt$uniformity$p.value),
    dispersion_p = as.numeric(tt$dispersion$p.value),
    outliers_p   = as.numeric(tt$outliers$p.value)
  )
})) %>%
  mutate(across(c(KS_p, dispersion_p, outliers_p), ~ signif(.x, 3)))
print(as.data.frame(dh_tab))

cat("\nDHARMa verdict (alpha = 0.05):\n")
for (nm in nut_order) {
  t <- dh_tab %>% filter(nutrient == nm)
  issues <- c()
  if (t$KS_p < 0.05)         issues <- c(issues, "residual non-uniformity")
  if (t$dispersion_p < 0.05) issues <- c(issues, "dispersion misspecification")
  if (t$outliers_p < 0.05)   issues <- c(issues, "outlier structure")
  cat(sprintf("  %-10s: %s\n", nut_lab[[nm]],
              if (length(issues)) paste(issues, collapse = "; ") else "no issues detected"))
}

    nutrient   KS_p dispersion_p outliers_p
1     Carbon 0.0645        0.958          1
2   Nitrogen 0.2040        0.800          1
3 Phosphorus 0.9160        0.508          1



DHARMa verdict (alpha = 0.05):


  SOC       : no issues detected
  Total N   : no issues detected
  Total P   : no issues detected


## 8 - Interpretation summary (auto-generated)


In [11]:
for (nm in nut_order) {
  a  <- anova_tab %>% filter(nutrient == nm, term != "(Intercept)")
  ip <- a$p_value[a$term == "treatment:period"]
  ta <- dh_tab %>% filter(nutrient == nm)
  rr <- r2_tab %>% filter(nutrient == nm)
  pt <- pair_tab %>% filter(nutrient == nm)
  ct <- emm_tab %>% filter(nutrient == nm)

  cat("==============================================================\n")
  cat(sprintf("NUTRIENT: %s (%s; response = %s)\n",
              nut_lab[[nm]], nm, unique(soil$response_name[soil$nutrient == nm])))

  tp <- a$p_value[a$term == "treatment"]
  cat(sprintf("- Treatment main effect: p = %.4g -> %s\n",
              tp, ifelse(tp < 0.05, "significant", "not significant")))

  prw <- a %>% filter(term != "treatment:period", grepl("period", term))
  if (nrow(prw) == 1) {
    cat(sprintf("- Period main effect: p = %.4g -> %s\n",
                prw$p_value, ifelse(prw$p_value < 0.05, "significant", "not significant")))
  } else {
    cat("- Period main effect (component terms): ")
    for (i in seq_len(nrow(prw))) cat(sprintf("%s p=%.4g; ", prw$term[i], prw$p_value[i]))
    cat("\n")
  }

  cat(sprintf("- Interaction (treatment:period): p = %s -> %s\n",
              ifelse(is.na(ip), "NA", sprintf("%.4g", ip)),
              ifelse(is.na(ip), "not estimable",
                     ifelse(ip < 0.05, "significant; interpret treatments period-by-period",
                            "not significant; treatment and period act additively (see EMms and model comparison)"))))

  cat(sprintf("- EMM range across cells: %.3f-%.3f (response scale)\n", min(ct$mean), max(ct$mean)))

  cat(sprintf("- Treatment ratios (%s) per period:\n", trt_cmp[[nm]]))
  for (i in seq_len(nrow(pt))) {
    cat(sprintf("    %s: ratio = %.3f [%.3f, %.3f], adj p = %.4g -> %s\n",
                pt$period[i], pt$ratio[i], pt$LCL[i], pt$UCL[i], pt$p.value[i],
                ifelse(pt$p.value[i] < 0.05, "significant", "not significant")))
  }

  cat(sprintf("- Explained variation: R2 marginal = %s, R2 conditional = %s (link-scale ICC above)\n",
              ifelse(is.na(rr$R2_marginal), "NA", sprintf("%.3f", rr$R2_marginal)),
              ifelse(is.na(rr$R2_conditional), "NA", sprintf("%.3f", rr$R2_conditional))))

  cat(sprintf("- DHARMa: KS p = %s, dispersion p = %s, outlier p = %s -> %s\n",
              ta$KS_p, ta$dispersion_p, ta$outliers_p,
              ifelse(all(c(ta$KS_p, ta$dispersion_p, ta$outliers_p) >= 0.05),
                     "residuals consistent with the Gamma(log) GLMM",
                     "one or more flags - the LMM sensitivity note applies")))
  cat("\n")
}

NUTRIENT: SOC (Carbon; response = soc)
- Treatment main effect: p = 0.9286 -> not significant
- Period main effect: p = 0.0004328 -> significant
- Interaction (treatment:period): p = 0.5399 -> not significant; treatment and period act additively (see EMms and model comparison)
- EMM range across cells: 2.500-3.940 (response scale)
- Treatment ratios (C/I) per period:
    Week_0: ratio = 0.991 [0.808, 1.210], adj p = 0.929 -> not significant
    Week_3: ratio = 0.886 [0.723, 1.090], adj p = 0.244 -> not significant
    Week_6: ratio = 0.818 [0.666, 1.000], adj p = 0.0544 -> not significant
    Week_9: ratio = 0.970 [0.792, 1.190], adj p = 0.772 -> not significant
- Explained variation: R2 marginal = NA, R2 conditional = NA (link-scale ICC above)
- DHARMa: KS p = 0.0645, dispersion p = 0.958, outlier p = 1 -> residuals consistent with the Gamma(log) GLMM

NUTRIENT: Total N (Nitrogen; response = total_nitrogen)
- Treatment main effect: p = 0.2011 -> not significant
- Period main effect: p

In [12]:
model_pack <- list(
  fits      = fits,
  variants  = variants,
  emm       = emm_list,
  sims      = sims,
  fx_tab    = fx_tab,
  vc_tab    = vc_tab,
  r2_tab    = r2_tab,
  anova_tab = anova_tab,
  cmp_tab   = cmp_tab,
  emm_tab   = emm_tab,
  pair_tab  = pair_tab,
  dh_tab    = dh_tab
)
saveRDS(model_pack, "soil_glmm_models.rds")
cat("Saved: soil_glmm_models.rds\n")
cat("\n--- Session info (appendix) ---\n")
sessionInfo()

Saved: soil_glmm_models.rds



--- Session info (appendix) ---


R version 4.6.0 (2026-04-24)
Platform: aarch64-apple-darwin23
Running under: macOS Tahoe 26.6.2

Matrix products: default
BLAS:   /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.0.dylib 
LAPACK: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRlapack.dylib;  LAPACK version 3.12.1

locale:
[1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8

time zone: Asia/Manila
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] AICcmodavg_2.3-4   DHARMa_0.5.0       performance_0.19.0 car_3.1-5         
 [5] carData_3.0-6      emmeans_2.0.4      ggplot2_4.0.3      tidyr_1.3.2       
 [9] dplyr_1.2.1        lme4_2.0-6         Matrix_1.7-5      

loaded via a namespace (and not attached):
 [1] generics_0.1.4     lattice_0.22-9     digest_0.6.39      magrittr_2.0.5    
 [5] evaluate_1.0.5     grid_4.6.0         estimability_2.0.0 RColorBrewer_1.1-3
 [9] pbdZMQ_0.3-14      mvt